# 🏥 Healthcare Veri Seti — MVP v2

**Veri Seti:** [Kaggle — Healthcare Dataset (prasad22)](https://www.kaggle.com/datasets/prasad22/healthcare-dataset)  
**Hedef:** Hasta test sonucunu tahmin etmek (Normal / Abnormal / Inconclusive)

## Proje Akışı
1. Kütüphane ve Veri Yükleme  
2. Keşifsel Veri Analizi (EDA)  
3. Veri Ön İşleme  
4. Model Eğitimi ve Test Skoru  
5. Model Kaydetme  
6. Streamlit Uygulaması (UI + Deploy)

## 1. Kütüphane ve Veri Yükleme

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print('Kütüphaneler yüklendi.')

In [ ]:
df = pd.read_csv('../data/healthcare_dataset.csv')

print(f'Satır sayısı : {df.shape[0]:,}')
print(f'Sütun sayısı : {df.shape[1]}')
df.head()

## 2. Keşifsel Veri Analizi (EDA)

In [ ]:
print('=== Değişken Tipleri & Eksik Değerler ===')
info = pd.DataFrame({
    'dtype'       : df.dtypes,
    'null_sayisi' : df.isnull().sum(),
    'null_yuzde'  : (df.isnull().sum() / len(df) * 100).round(2)
})
print(info)

In [ ]:
print('=== Sayısal Değişkenler ===')
print(df.describe())

In [ ]:
print('=== Kategorik Değişkenler ===')
cat_cols = ['Gender', 'Blood Type', 'Medical Condition',
            'Insurance Provider', 'Admission Type', 'Medication', 'Test Results']

for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts())

## 3. Veri Ön İşleme

In [ ]:
df_model = df.copy()

# Tarih feature'ları
df_model['Date of Admission'] = pd.to_datetime(df_model['Date of Admission'])
df_model['Discharge Date']    = pd.to_datetime(df_model['Discharge Date'])
df_model['yatis_gun']         = (df_model['Discharge Date'] - df_model['Date of Admission']).dt.days
df_model['admission_ay']      = df_model['Date of Admission'].dt.month

# Label Encoding
le = LabelEncoder()
encode_cols = ['Gender', 'Blood Type', 'Medical Condition',
               'Insurance Provider', 'Admission Type', 'Medication']
for col in encode_cols:
    df_model[col + '_enc'] = le.fit_transform(df_model[col])

# Hedef değişken
df_model['test_target'] = le.fit_transform(df_model['Test Results'])
target_map = dict(zip(le.transform(le.classes_), le.classes_))
print('Sınıf eşleştirmesi:', target_map)

# Feature listesi
features = ['Age', 'yatis_gun', 'Billing Amount', 'admission_ay',
            'Gender_enc', 'Blood Type_enc', 'Medical Condition_enc',
            'Insurance Provider_enc', 'Admission Type_enc', 'Medication_enc']

X = df_model[features]
y = df_model['test_target']

print(f'\nFeature sayısı : {X.shape[1]}')
print(f'Örnek sayısı   : {X.shape[0]:,}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Eğitim seti: {X_train.shape[0]:,} satır')
print(f'Test seti  : {X_test.shape[0]:,} satır')

## 4. Model Eğitimi ve Test Skoru

**Random Forest Classifier** — birden fazla karar ağacının oylamasıyla tahmin üretir. Yüksek doğruluk, düşük overfitting.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)

print('=== Test Sonuçları ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=list(target_map.values())))

## 5. Model Kaydetme

In [ ]:
joblib.dump(model,  'model_classification.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('model_classification.pkl → kaydedildi')
print('scaler.pkl              → kaydedildi')